# 🧠 Apoyo — Clasificación de texto

Ejecuta estos ejemplos después de crear el ambiente del examen con Python 3.12 y declarar `scikit-learn` con `uv add scikit-learn`. Usa el kernel de ese ambiente. Los ejemplos emplean sólo frases inventadas y no descargan datos.

Este apoyo presenta un camino de referencia para adaptar un sistema de ML a texto. Puedes elegir otras representaciones, familias y frameworks si justificas tus decisiones y cumples los resultados del README. Cada ejemplo es autónomo y vuelve a importar lo que necesita.


## 1. 🔤 Del texto a números: TF-IDF

Un clasificador necesita números. `TfidfVectorizer` aprende un vocabulario y asigna pesos a sus términos. Palabras frecuentes en casi todos los documentos suelen aportar menos capacidad de distinguirlos. Una matriz dispersa almacena principalmente los valores distintos de cero para ahorrar memoria.

`fit_transform` aprende y transforma los textos de entrenamiento. `transform` sólo aplica lo aprendido. Un término completamente nuevo no crea una columna durante inferencia. El siguiente ejemplo muestra tres documentos, las columnas aprendidas y que el número de columnas se conserva al transformar texto nuevo. `get_feature_names_out()` devuelve el vocabulario; `.shape` devuelve filas y columnas.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

entrenamiento = ["class meeting tomorrow", "free prize today", "class project meeting"]
vectorizador = TfidfVectorizer()
matriz = vectorizador.fit_transform(entrenamiento)
nuevo = vectorizador.transform(["class unknownword"])
print("Vocabulario:", vectorizador.get_feature_names_out().tolist())
print("Entrenamiento:", matriz.shape)
print("Texto nuevo:", nuevo.shape)
assert matriz.shape == (3, 7)
assert nuevo.shape[1] == matriz.shape[1]
assert "unknownword" not in vectorizador.vocabulary_


## 2. 🛠️ Pipeline y dos familias de clasificadores

`Pipeline` encadena pasos con nombre: el primero transforma el texto y el segundo clasifica. `fit` ajusta ambos con entrenamiento; `predict` transforma y predice sin reajustar. La regresión logística aprende pesos discriminativos; Multinomial Naive Bayes combina evidencia de términos bajo una suposición simplificadora de independencia condicional. Ninguno comprende el mensaje como una persona.

La lista de etiquetas es el objetivo, separado de los textos. `C` es la inversa de la fuerza de regularización en regresión logística. `alpha` suaviza las frecuencias en Naive Bayes. No confundir dos configuraciones de un mismo algoritmo con dos familias.

El resultado esperado son dos etiquetas por modelo, siempre dentro de `ham`/`spam`. El conjunto es deliberadamente diminuto: no sirve para demostrar calidad predictiva.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

textos = [
    "class meeting tomorrow", "project meeting after class", "bring the project notes",
    "free prize claim now", "win a free prize", "claim your prize today",
]
etiquetas = ["ham", "ham", "ham", "spam", "spam", "spam"]
modelos = {
    "logistic-regression": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "multinomial-nb": MultinomialNB(alpha=1.0),
}
for nombre, clasificador in modelos.items():
    pipeline = Pipeline([("tfidf", TfidfVectorizer()), ("classifier", clasificador)])
    pipeline.fit(textos, etiquetas)
    predicciones = pipeline.predict(["class project meeting", "claim free prize"])
    print(nombre, predicciones.tolist())
    assert len(predicciones) == 2
    assert set(predicciones) <= {"ham", "spam"}


## 3. 📊 Métricas y errores

Si `spam` es la clase positiva: TP es spam detectado, FP es ham marcado como spam y FN es spam no detectado. Precision = TP/(TP+FP); recall = TP/(TP+FN); F1 = 2·precision·recall/(precision+recall). `zero_division=0` define el resultado cuando una división no tiene denominador útil; no oculta el motivo, que debes explicar.

En este ejemplo hay 3 ham y 1 spam. Predecir siempre ham obtiene accuracy 0.75 y F1 de spam 0. El modelo de ejemplo detecta el spam, pero marca también un ham: precision 0.5, recall 1 y F1 aproximadamente 0.667. La matriz usa filas reales y columnas predichas, en orden `ham`, `spam`.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

reales = ["ham", "ham", "ham", "spam"]
entradas = [[0], [1], [2], [3]]
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(entradas, reales)
pred_base = baseline.predict(entradas)
pred_modelo = ["ham", "ham", "spam", "spam"]
print("Baseline accuracy:", accuracy_score(reales, pred_base))
print("Baseline F1 spam:", f1_score(reales, pred_base, pos_label="spam", zero_division=0))
print("Precision spam:", precision_score(reales, pred_modelo, pos_label="spam"))
print("Recall spam:", recall_score(reales, pred_modelo, pos_label="spam"))
print("F1 spam:", round(f1_score(reales, pred_modelo, pos_label="spam"), 3))
print("Matriz ham/spam:\n", confusion_matrix(reales, pred_modelo, labels=["ham", "spam"]))
assert accuracy_score(reales, pred_base) == 0.75
assert f1_score(reales, pred_base, pos_label="spam", zero_division=0) == 0


## 4. 🧭 Cómo llevarlo al examen

En el camino de este apoyo, separa los datos antes de ajustar TF-IDF. En cada trial crea un pipeline nuevo, ajústalo con entrenamiento y evalúa sobre validación. Justifica en tu solución la métrica de selección y el mecanismo de coordinación. Cierra configuración y champion antes de consultar prueba; evalúala sin regresar al tuning. Registra representación y modelo de forma reproducible. La API sólo carga y predice.

Las frases anteriores son ejemplos didácticos, no un conjunto de evaluación. En el examen usa las particiones y comprobaciones especificadas en el README.

## 🧠 Recopilación

El texto puede transformarse con un vocabulario aprendido y reutilizarse sin reajustar. El pipeline conserva esa relación durante entrenamiento y servicio. Dos métricas pueden contar historias distintas: compara con una referencia e interpreta qué errores tienen consecuencias para quien usa la aplicación.

## ✅ Comprobación

- Puedo distinguir `fit_transform` de `transform` y explicar dónde se aprende el vocabulario.
- Puedo identificar FP y FN con `spam` como clase positiva.
- Puedo explicar por qué un baseline de clase mayoritaria puede engañar si sólo miro accuracy.

## 📚 Referencias

- [Representación de texto](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [Multinomial Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html)
- [Regresión logística](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Métricas de clasificación](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
